# 🎙️ Voice Over Studio (F5-TTS Indonesia) — Google Colab

Jalankan backend voice cloning berbahasa Indonesia **gratis di GPU Colab**, lalu hubungkan frontend `index.html` ke URL publik yang dihasilkan.

**Langkah:**
1. Menu **Runtime → Change runtime type → T4 GPU**, lalu Save.
2. Jalankan sel di bawah berurutan (Shift+Enter).
3. Salin URL `https://xxxx.trycloudflare.com` yang muncul ke kotak **⚙️ Server** di frontend.

> Model: F5-TTS fine-tune Bahasa Indonesia (`Eempostor/F5-TTS-INDO-FINETUNE-V2`). Pakai hanya untuk suara Anda sendiri / narator yang sudah mengizinkan.

In [ ]:
# 1) Pasang dependensi
!pip -q install f5-tts huggingface_hub fastapi "uvicorn[standard]" python-multipart
!apt-get -qq install -y ffmpeg > /dev/null
print('Selesai.')

In [ ]:
# 2) Ambil kode backend + frontend.
#    Ganti URL di bawah dengan repo Anda. Jika repo PRIVAT, unggah folder
#    'voiceover/' secara manual lewat panel Files di kiri, lalu lewati sel ini.
REPO = 'https://github.com/syahrulq78/portal-bmp.git'
BRANCH = 'claude/voice-cloning-voiceover-app-m622ri'
import os
if not os.path.exists('portal-bmp'):
    !git clone --branch $BRANCH --depth 1 $REPO 2>/dev/null || echo 'Clone gagal (mungkin repo privat). Unggah folder voiceover/ manual.'
%cd /content/portal-bmp/voiceover/backend 2>/dev/null || %cd /content

In [ ]:
# 3) Siapkan tunnel publik (cloudflared)
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared
print('cloudflared siap.')

In [ ]:
# 4) Jalankan server + tampilkan URL publik
import subprocess, time, re

# Jalankan uvicorn (app.py harus ada di folder ini). Proses pertama akan
# mengunduh checkpoint F5-TTS Indonesia saat permintaan sintesis pertama.
server = subprocess.Popen(['uvicorn', 'app:app', '--host', '0.0.0.0', '--port', '8000'])
time.sleep(4)

tunnel = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in tunnel.stdout:
    print(line, end='')
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        print('\n\n==============================================')
        print(' SALIN URL INI KE KOTAK SERVER DI FRONTEND:')
        print(' ', m.group(0))
        print('==============================================\n')
        break
# Biarkan sel ini tetap berjalan agar server hidup.